# Project Summary

## Project Title
Practo Clinic Appointment Follow-up Analyzer

## Business Problem
Clinic operations data is often spread across multiple files such as appointments, patients, doctors, follow-ups, complaints, tickets, and availability records.

This makes it difficult for operations teams to quickly identify:
- missed appointments
- overdue patient follow-ups
- unresolved complaints
- service ticket SLA breaches
- repeated no-shows
- scheduling pressure
- high-risk cases requiring immediate attention

## Project Objective
The objective of this project is to combine and analyze clinic operational data so that each appointment can be evaluated for operational risk, assigned a priority, given a patient impact score, and mapped to a recommended next action.

## Target Users
- Clinic Operations Team
- Patient Support Team
- Doctor Coordination Team
- Escalation Team
- Operations Managers

## Technologies Used
- Google Colab
- Python
- Pandas
- CSV
- Excel
- JSON
- TXT
- ipywidgets

## GitHub Repository

Project Repository:  
https://github.com/reloaddd/practo-clinic-followup-analyzer


# Practo Clinic Appointment Follow-up Analyzer

This project is a clinic appointment follow-up analyzer for clinic operations and patient-support teams.

The main problem is that appointment, follow-up, complaint, ticket, patient, and doctor information is spread across multiple files.

The goal is to identify missed appointments, overdue follow-ups, service issues, high-risk cases, and the next recommended action.

The solution is implemented completely inside Google Colab using Python and Pandas with a lightweight widget-based interface.

I intentionally avoided databases, web frameworks, ML models, and external APIs because they are outside the project scope.

# 1. Import Libraries

In [ ]:
import pandas as pd
import os
import json
from IPython.display import display, clear_output
import ipywidgets as widgets

pd.set_option("display.max_columns", None)

# 2. Multi-File Data Loader

The project contains multiple file formats, so the loader handles CSV, Excel, JSON, and TXT.

Appointments are the central operational dataset.

Patients provide patient profile and risk information.

Doctors and doctor availability provide doctor and slot information.

Follow-ups, complaints, and service tickets provide the operational issues associated with appointments.

I also distinguish critical files from optional files so the notebook can handle missing inputs gracefully.


In [ ]:
from google.colab import files

print("Upload the PRD input files.")
uploaded = files.upload()

Upload the PRD input files.


Saving patients.csv to patients.csv
Saving service_tickets.csv to service_tickets.csv
Saving appointments.csv to appointments.csv
Saving clinic_notes.txt to clinic_notes.txt
Saving doctor_availability.csv to doctor_availability.csv
Saving doctors.csv to doctors.csv
Saving followups.xlsx to followups.xlsx
Saving patient_complaints.json to patient_complaints.json


In [ ]:
critical_files = ["appointments.csv", "patients.csv", "doctors.csv"]
optional_files = [
    "doctor_availability.csv", "followups.xlsx",
    "patient_complaints.json", "service_tickets.csv",
    "clinic_notes.txt"
]

missing = [f for f in critical_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError("Missing critical files: " + ", ".join(missing))

for f in optional_files:
    if not os.path.exists(f):
        print("Warning - optional file missing:", f)

appointments = pd.read_csv("appointments.csv")
patients = pd.read_csv("patients.csv")
doctors = pd.read_csv("doctors.csv")
doctor_availability = pd.read_csv("doctor_availability.csv") if os.path.exists("doctor_availability.csv") else pd.DataFrame()
followups = pd.read_excel("followups.xlsx") if os.path.exists("followups.xlsx") else pd.DataFrame()
complaints = pd.read_json("patient_complaints.json") if os.path.exists("patient_complaints.json") else pd.DataFrame()
service_tickets = pd.read_csv("service_tickets.csv") if os.path.exists("service_tickets.csv") else pd.DataFrame()
clinic_notes = open("clinic_notes.txt", encoding="utf-8").read() if os.path.exists("clinic_notes.txt") else ""

print("Appointments:", appointments.shape)
print("Patients:", patients.shape)
print("Doctors:", doctors.shape)
print("Availability:", doctor_availability.shape)
print("Follow-ups:", followups.shape)
print("Complaints:", complaints.shape)
print("Tickets:", service_tickets.shape)

Appointments: (4515, 21)
Patients: (1200, 13)
Doctors: (650, 11)
Availability: (1400, 10)
Follow-ups: (1300, 14)
Complaints: (1200, 12)
Tickets: (1300, 13)


In [ ]:
patients

,patient_id,full_name,age,gender,city,preferred_language,patient_segment,primary_condition,risk_level,registration_date,phone_masked,email_masked,active_status
0,PT000001,Saanvi Reddy,47,Female,Pune,Hindi,Regular,ENT,Low,2024-06-14,XXXXXX4582,patient1@synthetic.example,Active
1,PT000002,Priya Reddy,36,Male,Lucknow,English,Senior Care,Orthopedic Pain,Low,2022-08-10,XXXXXX6574,patient2@synthetic.example,Active
2,PT000003,Kriti Das,50,Female,Chennai,Hindi,Chronic Care,Cardiac Follow-up,High,2021-11-18,XXXXXX5803,patient3@synthetic.example,Active
3,PT000004,Aarohi Ghosh,64,Male,Bengaluru,Tamil,Chronic Care,Cardiac Follow-up,Medium,2022-06-08,XXXXXX6977,patient4@synthetic.example,Active
4,PT000005,Kriti Mishra,35,Female,Mumbai,Tamil,New Patient,ENT,Low,2021-11-08,XXXXXX4598,patient5@synthetic.example,Active
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,PT001196,Anika Das,46,Female,Chennai,Bengali,Regular,Gastroenterology,Low,2024-03-10,XXXXXX6783,patient1196@synthetic.example,Active
1196,PT001197,Arnav Sethi,38,Other,Delhi NCR,Bengali,Chronic Care,Hypertension,Medium,2023-11-20,XXXXXX9948,patient1197@synthetic.example,Inactive
1197,PT001198,Diya Malhotra,48,Male,Chennai,Gujarati,New Patient,Diabetes,Medium,2022-10-18,XXXXXX3747,patient1198@synthetic.example,Inactive
1198,PT001199,Riya Kulkarni,42,Other,Hyderabad,Marathi,New Patient,Mental Wellness,Low,2022-02-23,XXXXXX2740,patient1199@synthetic.example,Active


# 3. Data Validation Engine

Before analyzing anything, I validate the raw data because incorrect data can produce incorrect business conclusions.

I check required columns and unexpected extra columns.

I check duplicate primary IDs.

I check null values in important fields such as appointment ID, patient ID, doctor ID, and appointment date.

I detect invalid dates.

I check orphan patient, doctor, follow-up, complaint, and ticket references.

For doctor availability, I also detect impossible or overbooked slot records.

example: “If an appointment contains a doctor ID that does not exist in the doctor table, I flag it rather than silently trusting the relationship.”

In [ ]:
required_columns = {
"appointments": ["appointment_id","booking_id","patient_id","doctor_id","clinic_id","appointment_date","scheduled_start_time","scheduled_end_time","booking_created_at","booking_channel","consultation_mode","appointment_status","check_in_status","cancellation_reason","consultation_fee","fee_paid","payment_status","service_region","patient_issue_flag","previous_no_show_count","notes_tag"],
"patients": ["patient_id","full_name","age","gender","city","preferred_language","patient_segment","primary_condition","risk_level","registration_date","phone_masked","email_masked","active_status"],
"doctors": ["doctor_id","doctor_name","specialty","clinic_id","clinic_city","service_region","consultation_mode","standard_slot_minutes","doctor_rating","active_status","joining_date"],
"doctor_availability": ["availability_id","doctor_id","clinic_id","availability_date","shift","total_slots","booked_slots","blocked_slots","doctor_status","service_region"],
"followups": ["followup_id","appointment_id","patient_id","doctor_id","followup_type","followup_due_date","followup_status","followup_outcome","contact_attempts","last_contact_channel","assigned_team","risk_reason","next_action","created_at"],
"complaints": ["complaint_id","appointment_id","patient_id","complaint_date","complaint_category","complaint_text","complaint_severity","sentiment","escalated_flag","repeated_issue_flag","source_channel","resolved_status"],
"service_tickets": ["ticket_id","appointment_id","patient_id","ticket_created_at","ticket_type","ticket_status","sla_hours","hours_open","sla_breached","escalation_level","assigned_queue","last_updated_at","resolution_notes"]
}

datasets = {"appointments":appointments,"patients":patients,"doctors":doctors,"doctor_availability":doctor_availability,"followups":followups,"complaints":complaints,"service_tickets":service_tickets}
validation_rows=[]
for name, df in datasets.items():
    if df.empty:
        validation_rows.append({"dataset":name,"check":"file/data","status":"Warning","details":"Empty or not supplied"})
        continue
    missing_cols=[x for x in required_columns[name] if x not in df.columns]
    extra_cols=[x for x in df.columns if x not in required_columns[name]]
    validation_rows.append({"dataset":name,"check":"required columns","status":"Pass" if not missing_cols else "Fail","details":", ".join(missing_cols) if missing_cols else "None"})
    validation_rows.append({"dataset":name,"check":"extra columns","status":"Info" if extra_cols else "Pass","details":", ".join(extra_cols) if extra_cols else "None"})

validation_report=pd.DataFrame(validation_rows)
display(validation_report)

,dataset,check,status,details
0,appointments,required columns,Pass,None
1,appointments,extra columns,Pass,None
2,patients,required columns,Pass,None
3,patients,extra columns,Pass,None
4,doctors,required columns,Pass,None
5,doctors,extra columns,Pass,None
6,doctor_availability,required columns,Pass,None
7,doctor_availability,extra columns,Pass,None
8,followups,required columns,Pass,None
9,followups,extra columns,Pass,None


In [ ]:
id_checks=[("appointments",appointments,"appointment_id"),("patients",patients,"patient_id"),("doctors",doctors,"doctor_id"),("doctor_availability",doctor_availability,"availability_id"),("followups",followups,"followup_id"),("complaints",complaints,"complaint_id"),("service_tickets",service_tickets,"ticket_id")]
duplicate_rows=[]
for name,df,col in id_checks:
    if not df.empty and col in df.columns:
        duplicate_rows.append({"dataset":name,"id_column":col,"duplicate_rows":int(df[col].duplicated(keep=False).sum())})
display(pd.DataFrame(duplicate_rows))

critical_nulls=[]
for name, cols in {"appointments":["appointment_id","patient_id","doctor_id","appointment_date"],"patients":["patient_id","full_name"],"doctors":["doctor_id","doctor_name"]}.items():
    for col in cols:
        critical_nulls.append({"dataset":name,"column":col,"null_count":int(datasets[name][col].isna().sum())})
display(pd.DataFrame(critical_nulls))

,dataset,id_column,duplicate_rows
0,appointments,appointment_id,30
1,patients,patient_id,0
2,doctors,doctor_id,0
3,doctor_availability,availability_id,0
4,followups,followup_id,0
5,complaints,complaint_id,0
6,service_tickets,ticket_id,0


,dataset,column,null_count
0,appointments,appointment_id,0
1,appointments,patient_id,0
2,appointments,doctor_id,0
3,appointments,appointment_date,8
4,patients,patient_id,0
5,patients,full_name,0
6,doctors,doctor_id,0
7,doctors,doctor_name,0


In [ ]:
date_test=pd.to_datetime(appointments["appointment_date"],errors="coerce")
invalid_appointment_dates=appointments[date_test.isna()]
orphan_patient_records=appointments[~appointments["patient_id"].isin(patients["patient_id"])]
orphan_doctor_records=appointments[~appointments["doctor_id"].isin(doctors["doctor_id"])]

print("Invalid appointment dates:",len(invalid_appointment_dates))
print("Orphan patient references:",len(orphan_patient_records))
print("Orphan doctor references:",len(orphan_doctor_records))

if not doctor_availability.empty:
    total=pd.to_numeric(doctor_availability["total_slots"],errors="coerce").fillna(0)
    booked=pd.to_numeric(doctor_availability["booked_slots"],errors="coerce").fillna(0)
    blocked=pd.to_numeric(doctor_availability["blocked_slots"],errors="coerce").fillna(0)
    overbooked_availability=doctor_availability[(booked+blocked)>total]
else:
    overbooked_availability=pd.DataFrame()
print("Overbooked availability records:",len(overbooked_availability))

Invalid appointment dates: 19
Orphan patient references: 0
Orphan doctor references: 0
Overbooked availability records: 252


# 4. Data Cleaning and Standardization

After validation, I create clean copies so the raw datasets remain unchanged.

I standardize text spacing and status values.

For example, No Show, NO SHOW, and similar variations become a consistent representation.

I convert date columns to Pandas datetime.

I convert fees, counts, SLA hours, and similar fields to numeric data.

This standardization is important because business rules cannot work reliably on inconsistent values.

In [ ]:
appointments_clean=appointments.copy(); patients_clean=patients.copy(); doctors_clean=doctors.copy()
doctor_availability_clean=doctor_availability.copy(); followups_clean=followups.copy(); complaints_clean=complaints.copy(); service_tickets_clean=service_tickets.copy()

def clean_status(s):
    return s.astype("string").fillna("").str.strip().str.lower().str.replace(r"[ -]+","_",regex=True)

for col in ["appointment_status","check_in_status","payment_status","consultation_mode","patient_issue_flag"]:
    appointments_clean[col]=clean_status(appointments_clean[col])
patients_clean["risk_level"]=clean_status(patients_clean["risk_level"])

appointments_clean["appointment_date"]=pd.to_datetime(appointments_clean["appointment_date"],errors="coerce")
appointments_clean["booking_created_at"]=pd.to_datetime(appointments_clean["booking_created_at"],errors="coerce")
for col in ["consultation_fee","fee_paid","previous_no_show_count"]:
    appointments_clean[col]=pd.to_numeric(appointments_clean[col],errors="coerce")

if not followups_clean.empty:
    followups_clean["followup_status"]=clean_status(followups_clean["followup_status"])
    followups_clean["followup_due_date"]=pd.to_datetime(followups_clean["followup_due_date"],errors="coerce")
    followups_clean["created_at"]=pd.to_datetime(followups_clean["created_at"],errors="coerce")
    followups_clean["contact_attempts"]=pd.to_numeric(followups_clean["contact_attempts"],errors="coerce")
if not complaints_clean.empty:
    for col in ["complaint_severity","sentiment","resolved_status"]: complaints_clean[col]=clean_status(complaints_clean[col])
    complaints_clean["complaint_date"]=pd.to_datetime(complaints_clean["complaint_date"],errors="coerce")
if not service_tickets_clean.empty:
    service_tickets_clean["ticket_status"]=clean_status(service_tickets_clean["ticket_status"])
    service_tickets_clean["ticket_created_at"]=pd.to_datetime(service_tickets_clean["ticket_created_at"],errors="coerce")
    service_tickets_clean["last_updated_at"]=pd.to_datetime(service_tickets_clean["last_updated_at"],errors="coerce")
    for col in ["sla_hours","hours_open"]: service_tickets_clean[col]=pd.to_numeric(service_tickets_clean[col],errors="coerce")
if not doctor_availability_clean.empty:
    doctor_availability_clean["availability_date"]=pd.to_datetime(doctor_availability_clean["availability_date"],errors="coerce")
print("Cleaning completed.")

Cleaning completed.


# 5. Missed Appointment Analysis

This section converts the PRD’s missed-appointment rules into flags.

An explicit no-show or missed status is directly classified as missed.

A past booked or confirmed appointment without valid check-in evidence can be classified as likely missed.

Cancelled and rescheduled appointments are excluded from missed classification.

Invalid dates are treated as data-quality issues instead of being guessed.

This creates the missed flag that is later used in priority and impact scoring.

In [ ]:
today=pd.Timestamp.today().normalize()
status=appointments_clean["appointment_status"]
checkin=appointments_clean["check_in_status"]
appointments_clean["explicit_missed_flag"]=status.isin(["no_show","noshow","missed","missed_appointment"])
appointments_clean["valid_checkin_flag"]=checkin.isin(["checked_in","checkedin","arrived","present","completed"])
appointments_clean["likely_missed_flag"]=(appointments_clean["appointment_date"]<today)&status.isin(["booked","confirmed","scheduled","pending"])&~appointments_clean["valid_checkin_flag"]&~status.isin(["cancelled","canceled","rescheduled"])
appointments_clean["missed_flag"]=appointments_clean["explicit_missed_flag"]|appointments_clean["likely_missed_flag"]
appointments_clean["invalid_appointment_date_flag"]=appointments_clean["appointment_date"].isna()
appointments_clean["missed_type"]="Not Missed"
appointments_clean.loc[appointments_clean["likely_missed_flag"],"missed_type"]="Likely Missed Appointment"
appointments_clean.loc[appointments_clean["explicit_missed_flag"],"missed_type"]="Missed Appointment"
appointments_clean.loc[status.isin(["cancelled","canceled","rescheduled"]),"missed_type"]="Cancelled/Rescheduled"
appointments_clean.loc[appointments_clean["invalid_appointment_date_flag"],"missed_type"]="Data Quality Issue"
display(appointments_clean[["appointment_id","appointment_date","appointment_status","check_in_status","missed_type","missed_flag"]].head(20))

,appointment_id,appointment_date,appointment_status,check_in_status,missed_type,missed_flag
0,APP000001,2025-03-22,completed,checked_in,Not Missed,False
1,APP000002,2025-04-29,confirmed,,Likely Missed Appointment,True
2,APP000003,2025-02-22,confirmed,not_checked_in,Likely Missed Appointment,True
3,APP000004,2025-03-12,completed,checked_in,Not Missed,False
4,APP000005,2025-01-11,rescheduled,,Cancelled/Rescheduled,False
5,APP000006,2025-03-26,no_show,not_checked_in,Missed Appointment,True
6,APP000007,2025-04-06,completed,late_check_in,Not Missed,False
7,APP000008,2025-03-29,completed,checked_in,Not Missed,False
8,APP000009,2025-03-26,completed,checked_in,Not Missed,False
9,APP000010,2025-03-11,completed,checked_in,Not Missed,False


# 6. Doctor Availability Analysis

Here I analyze doctor capacity and slot pressure.

I calculate usable slots from total slots minus blocked slots.

I calculate slot utilization.

I flag overbooked shifts and high-utilization situations.

Then I aggregate the results doctor-wise.

This helps operations identify where patients may face scheduling or rescheduling problems.

In [ ]:
if not doctor_availability_clean.empty:
    for col in ["total_slots","booked_slots","blocked_slots"]:
        doctor_availability_clean[col]=pd.to_numeric(doctor_availability_clean[col],errors="coerce").fillna(0)
    doctor_availability_clean["usable_slots"]=(doctor_availability_clean["total_slots"]-doctor_availability_clean["blocked_slots"]).clip(lower=0)
    doctor_availability_clean["slot_utilization_pct"]=doctor_availability_clean.apply(lambda r: round(r["booked_slots"]/r["usable_slots"]*100,2) if r["usable_slots"]>0 else 0,axis=1)
    doctor_availability_clean["overbooked_shift_flag"]=(doctor_availability_clean["booked_slots"]+doctor_availability_clean["blocked_slots"])>doctor_availability_clean["total_slots"]
    doctor_availability_clean["slot_pressure_flag"]=(doctor_availability_clean["slot_utilization_pct"]>=90)|doctor_availability_clean["overbooked_shift_flag"]
    doctor_availability_summary=doctor_availability_clean.groupby("doctor_id").agg(total_slots=("total_slots","sum"),booked_slots=("booked_slots","sum"),blocked_slots=("blocked_slots","sum"),average_slot_utilization=("slot_utilization_pct","mean"),overbooked_shift_count=("overbooked_shift_flag","sum"),slot_pressure_flag=("slot_pressure_flag","max")).reset_index()
else:
    doctor_availability_summary=pd.DataFrame(columns=["doctor_id"])
display(doctor_availability_summary)

,doctor_id,total_slots,booked_slots,blocked_slots,average_slot_utilization,overbooked_shift_count,slot_pressure_flag
0,DR0001,26,19,1,92.065000,1,True
1,DR0002,64,41,9,69.282500,0,True
2,DR0003,24,15,4,75.000000,0,True
3,DR0004,54,49,4,100.733333,1,True
4,DR0006,14,19,0,135.710000,1,True
...,...,...,...,...,...,...,...
565,DR9954,8,6,0,75.000000,0,False
566,DR9968,10,6,0,60.000000,0,False
567,DR9974,8,1,0,12.500000,0,False
568,DR9975,20,15,2,83.330000,0,False


# 7. Patient Follow-up Analysis

This section checks whether follow-ups are completed or still pending.

I calculate the number of days overdue using the due date.

The logic separates normal pending follow-ups, delay warnings, and SLA breaches.

Three or more unsuccessful contact attempts are also treated as patient-contact risk.

Because one appointment can have multiple follow-up records, I aggregate them before merging into the master table.

In [ ]:
if not followups_clean.empty:
    followups_clean["followup_completed_flag"]=followups_clean["followup_status"].isin(["completed","closed","done","resolved"])
    followups_clean["days_overdue"]=(today-followups_clean["followup_due_date"]).dt.days.where(~followups_clean["followup_completed_flag"],0).fillna(0).clip(lower=0)
    followups_clean["followup_pending_flag"]=(~followups_clean["followup_completed_flag"])&followups_clean["days_overdue"].between(1,3)
    followups_clean["followup_delay_warning_flag"]=(~followups_clean["followup_completed_flag"])&followups_clean["days_overdue"].between(4,7)
    followups_clean["followup_sla_breach_flag"]=(~followups_clean["followup_completed_flag"])&(followups_clean["days_overdue"]>7)
    followups_clean["overdue_followup_flag"]=(~followups_clean["followup_completed_flag"])&(followups_clean["days_overdue"]>0)
    followups_clean["patient_contact_risk_flag"]=(~followups_clean["followup_completed_flag"])&(followups_clean["contact_attempts"].fillna(0)>=3)
    followup_summary=followups_clean.groupby("appointment_id").agg(followup_count=("followup_id","count"),followup_completed_flag=("followup_completed_flag","max"),overdue_followup_flag=("overdue_followup_flag","max"),followup_delay_warning_flag=("followup_delay_warning_flag","max"),followup_sla_breach_flag=("followup_sla_breach_flag","max"),patient_contact_risk_flag=("patient_contact_risk_flag","max"),max_days_overdue=("days_overdue","max"),max_contact_attempts=("contact_attempts","max")).reset_index()
    latest=followups_clean.sort_values("created_at").drop_duplicates("appointment_id",keep="last")[["appointment_id","followup_status","next_action"]].rename(columns={"followup_status":"latest_followup_status","next_action":"latest_followup_action"})
    followup_summary=followup_summary.merge(latest,on="appointment_id",how="left")
else:
    followup_summary=pd.DataFrame(columns=["appointment_id"])
display(followup_summary)

,appointment_id,followup_count,followup_completed_flag,overdue_followup_flag,followup_delay_warning_flag,followup_sla_breach_flag,patient_contact_risk_flag,max_days_overdue,max_contact_attempts,latest_followup_status,latest_followup_action
0,APP000004,1,False,True,False,True,True,525.0,4,pending,Confirm doctor slot
1,APP000005,1,False,True,False,True,True,526.0,4,pending,Call patient
2,APP000006,1,False,True,False,True,False,516.0,0,pending,Escalate to clinic manager
3,APP000011,1,False,True,False,True,True,540.0,5,pending,Close after confirmation
4,APP000014,1,True,False,False,False,False,0.0,4,completed,Call patient
...,...,...,...,...,...,...,...,...,...,...,...
1295,APP991296,1,False,True,False,True,True,631.0,3,overdue,Share prescription
1296,APP991297,1,False,True,False,True,True,591.0,3,overdue,Call patient
1297,APP991298,1,False,True,False,True,False,593.0,2,pending,Await patient response
1298,APP991299,1,True,False,False,False,False,0.0,1,completed,Await patient response


# 8. Service Ticket Analysis

Service tickets represent unresolved support issues connected with appointments.

I identify open, pending, in-progress, and escalated tickets.

I compare hours open against SLA hours to detect SLA breaches.

I detect L2 and L3 escalations.

I also identify slow resolutions where the case was eventually resolved but exceeded its SLA.

These features later influence case priority.

In [ ]:
if not service_tickets_clean.empty:
    service_tickets_clean["unresolved_ticket_flag"]=service_tickets_clean["ticket_status"].isin(["open","pending","in_progress","escalated"])
    supplied=service_tickets_clean["sla_breached"].astype("string").str.strip().str.lower().isin(["yes","true","1"])
    service_tickets_clean["ticket_sla_breach_flag"]=supplied|(service_tickets_clean["unresolved_ticket_flag"]&(service_tickets_clean["hours_open"]>service_tickets_clean["sla_hours"]))
    service_tickets_clean["slow_resolution_flag"]=service_tickets_clean["ticket_status"].isin(["resolved","closed","completed"])&(service_tickets_clean["hours_open"]>service_tickets_clean["sla_hours"])
    esc=service_tickets_clean["escalation_level"].astype("string").str.strip().str.lower()
    service_tickets_clean["ticket_escalated_flag"]=esc.isin(["l2","l3","2","3"])|(service_tickets_clean["ticket_status"]=="escalated")
    ticket_summary=service_tickets_clean.groupby("appointment_id").agg(ticket_count=("ticket_id","count"),unresolved_ticket_flag=("unresolved_ticket_flag","max"),ticket_sla_breach_flag=("ticket_sla_breach_flag","max"),slow_resolution_flag=("slow_resolution_flag","max"),ticket_escalated_flag=("ticket_escalated_flag","max"),max_hours_open=("hours_open","max")).reset_index()
    latest=service_tickets_clean.sort_values("last_updated_at").drop_duplicates("appointment_id",keep="last")[["appointment_id","ticket_status"]].rename(columns={"ticket_status":"latest_ticket_status"})
    ticket_summary=ticket_summary.merge(latest,on="appointment_id",how="left")
else:
    ticket_summary=pd.DataFrame(columns=["appointment_id"])
display(ticket_summary)

,appointment_id,ticket_count,unresolved_ticket_flag,ticket_sla_breach_flag,slow_resolution_flag,ticket_escalated_flag,max_hours_open,latest_ticket_status
0,APP000004,1,True,True,False,True,46,in_progress
1,APP000008,1,False,True,True,False,135,closed
2,APP000013,1,True,False,False,False,13,open
3,APP000016,1,True,False,False,True,3,open
4,APP000017,1,True,False,False,False,1,open
...,...,...,...,...,...,...,...,...
1295,APP771296,1,False,True,True,True,94,resolved
1296,APP771297,1,True,True,False,False,64,open
1297,APP771298,1,True,True,False,False,83,in_progress
1298,APP771299,1,True,True,False,False,136,open


# 9. Complaint Analysis

Complaints are linked to appointments and patients.

I identify unresolved complaints, serious complaints, angry sentiment, escalations, and repeated complaints.

I create both appointment-level and patient-level complaint summaries.

The patient-level summary helps identify repeated dissatisfaction across multiple appointments.

In [ ]:
if not complaints_clean.empty:
    complaints_clean["unresolved_complaint_flag"]=~complaints_clean["resolved_status"].isin(["resolved","closed","completed"])
    complaints_clean["serious_complaint_flag"]=complaints_clean["complaint_severity"].isin(["high","critical","severe"])
    complaints_clean["angry_complaint_flag"]=complaints_clean["sentiment"].isin(["angry","very_negative"])
    complaints_clean["complaint_escalated_flag"]=complaints_clean["escalated_flag"].astype("string").str.strip().str.lower().isin(["yes","true","1"])
    complaints_clean["repeated_complaint_flag"]=complaints_clean["repeated_issue_flag"].astype("string").str.strip().str.lower().isin(["yes","true","1"])
    complaint_summary=complaints_clean.groupby("appointment_id").agg(complaint_count=("complaint_id","count"),unresolved_complaint_flag=("unresolved_complaint_flag","max"),serious_complaint_flag=("serious_complaint_flag","max"),angry_complaint_flag=("angry_complaint_flag","max"),complaint_escalated_flag=("complaint_escalated_flag","max"),repeated_complaint_flag=("repeated_complaint_flag","max")).reset_index()
    patient_complaint_summary=complaints_clean.groupby("patient_id").agg(patient_complaint_count=("complaint_id","count"),patient_repeated_complaint_flag=("repeated_complaint_flag","max")).reset_index()
else:
    complaint_summary=pd.DataFrame(columns=["appointment_id"]); patient_complaint_summary=pd.DataFrame(columns=["patient_id"])
display(complaint_summary)

,appointment_id,complaint_count,unresolved_complaint_flag,serious_complaint_flag,angry_complaint_flag,complaint_escalated_flag,repeated_complaint_flag
0,APP000002,1,False,False,False,False,False
1,APP000004,1,False,False,False,False,False
2,APP000020,1,True,False,False,False,False
3,APP000021,1,False,False,False,True,False
4,APP000022,1,False,False,False,False,False
...,...,...,...,...,...,...,...
1195,APP881196,1,False,True,False,False,False
1196,APP881197,1,False,False,False,False,False
1197,APP881198,1,False,False,False,False,False
1198,APP881199,1,True,False,False,False,False


# 10. Duplicate and Data Anomaly Detection

I do not silently delete suspicious records.

Instead, I create flags for duplicate appointment IDs, possible duplicate bookings, orphan relationships, invalid dates, and payment anomalies.

This preserves traceability while still alerting the operations team.

The suspicious records are also exported in a separate anomaly report.

In [ ]:
appointments_clean["duplicate_appointment_id_flag"]=appointments_clean["appointment_id"].duplicated(keep=False)
appointments_clean["possible_duplicate_appointment_flag"]=appointments_clean.duplicated(["patient_id","doctor_id","appointment_date","scheduled_start_time"],keep=False)
appointments_clean["orphan_patient_flag"]=~appointments_clean["patient_id"].isin(patients_clean["patient_id"])
appointments_clean["orphan_doctor_flag"]=~appointments_clean["doctor_id"].isin(doctors_clean["doctor_id"])
appointments_clean["payment_anomaly_flag"]=(appointments_clean["payment_status"].isin(["paid","completed","success"])&(appointments_clean["fee_paid"].fillna(0)<=0))|((appointments_clean["fee_paid"]>appointments_clean["consultation_fee"])&appointments_clean["consultation_fee"].notna())
appointments_clean["data_anomaly_flag"]=appointments_clean[["duplicate_appointment_id_flag","possible_duplicate_appointment_flag","orphan_patient_flag","orphan_doctor_flag","payment_anomaly_flag","invalid_appointment_date_flag"]].any(axis=1)
anomaly_report=appointments_clean[appointments_clean["data_anomaly_flag"]].copy()
print("Appointment-level anomalies:",len(anomaly_report))
display(anomaly_report.head())

Appointment-level anomalies: 116


,appointment_id,booking_id,patient_id,doctor_id,clinic_id,appointment_date,scheduled_start_time,scheduled_end_time,booking_created_at,booking_channel,consultation_mode,appointment_status,check_in_status,cancellation_reason,consultation_fee,fee_paid,payment_status,service_region,patient_issue_flag,previous_no_show_count,notes_tag,explicit_missed_flag,valid_checkin_flag,likely_missed_flag,missed_flag,invalid_appointment_date_flag,missed_type,duplicate_appointment_id_flag,possible_duplicate_appointment_flag,orphan_patient_flag,orphan_doctor_flag,payment_anomaly_flag,data_anomaly_flag
16,APP000017,BK100017,PT000306,DR0473,CL037,2025-02-10,15:30,15:40,2025-01-29 12:30:00,App,in_clinic,completed,checked_in,NaN,699,749,paid,South,no,0,doctor_delay,False,True,False,False,False,Not Missed,False,False,False,False,True,True
18,APP000019,BK100019,PT000169,DR0034,CL047,2025-03-21,11:45,12:15,2025-03-21 07:45:00,Clinic Desk,in_clinic,completed,checked_in,NaN,1199,1249,paid,Central,no,0,normal,False,True,False,False,False,Not Missed,False,False,False,False,True,True
20,APP000021,BK100021,PT001096,DR0344,CL015,2025-01-17,11:00,11:10,2025-01-06 05:00:00,Web,in_clinic,completed,checked_in,NaN,699,799,paid,Central,yes,0,doctor_delay,False,True,False,False,False,Not Missed,False,False,False,False,True,True
67,APP000068,BK100068,PT000691,DR0108,CL014,NaT,20:30,20:50,2025-01-20 08:30:00,Partner Referral,video,booked,,NaN,1499,1499,paid,Central,no,1,NaN,False,False,False,False,True,Data Quality Issue,False,False,False,False,False,True
148,APP000149,BK100149,PT000196,DR0052,CL028,NaT,09:45,10:15,2025-03-19 22:45:00,Web,in_clinic,completed,late_check_in,NaN,1499,1499,paid,East,no,0,patient_confusion,False,False,False,False,True,Data Quality Issue,False,False,False,False,False,True


# 11. Final Appointment-Level Master Table

The master table creates one business-readable record per appointment.

I start from appointments and merge patient information using patient_id.

I merge doctor information using doctor_id.

Follow-up, complaint, and ticket tables are one-to-many, so I aggregate them first before merging.

This prevents duplicate appointment rows.

The final table contains patient, doctor, missed appointment, follow-up, complaint, ticket, availability, and anomaly features.

In [ ]:
master=appointments_clean.copy()
master=master.merge(patients_clean,on="patient_id",how="left",suffixes=("","_patient"))
doctor_cols=[x for x in ["doctor_id","doctor_name","specialty","clinic_city","standard_slot_minutes","doctor_rating"] if x in doctors_clean.columns]
master=master.merge(doctors_clean[doctor_cols].drop_duplicates("doctor_id"),on="doctor_id",how="left")
for summary,key in [(followup_summary,"appointment_id"),(complaint_summary,"appointment_id"),(patient_complaint_summary,"patient_id"),(ticket_summary,"appointment_id"),(doctor_availability_summary,"doctor_id")]:
    if not summary.empty: master=master.merge(summary,on=key,how="left")
for col in [x for x in master.columns if x.endswith("_flag")]: master[col]=master[col].fillna(False)
for col in [x for x in master.columns if x.endswith("_count")]: master[col]=pd.to_numeric(master[col],errors="coerce").fillna(0)
existing=set(followups_clean["appointment_id"]) if not followups_clean.empty else set()
master["followup_record_missing_flag"]=(master["missed_flag"]|(master["risk_level"]=="high"))&~master["appointment_id"].isin(existing)
print("Master table shape:",master.shape)
display(master.head())

Master table shape: (4515, 82)


/tmp/ipykernel_1821/787877995.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  for col in [x for x in master.columns if x.endswith("_flag")]: master[col]=master[col].fillna(False)


,appointment_id,booking_id,patient_id,doctor_id,clinic_id,appointment_date,scheduled_start_time,scheduled_end_time,booking_created_at,booking_channel,consultation_mode,appointment_status,check_in_status,cancellation_reason,consultation_fee,fee_paid,payment_status,service_region,patient_issue_flag,previous_no_show_count,notes_tag,explicit_missed_flag,valid_checkin_flag,likely_missed_flag,missed_flag,invalid_appointment_date_flag,missed_type,duplicate_appointment_id_flag,possible_duplicate_appointment_flag,orphan_patient_flag,orphan_doctor_flag,payment_anomaly_flag,data_anomaly_flag,full_name,age,gender,city,preferred_language,patient_segment,primary_condition,risk_level,registration_date,phone_masked,email_masked,active_status,doctor_name,specialty,clinic_city,standard_slot_minutes,doctor_rating,followup_count,followup_completed_flag,overdue_followup_flag,followup_delay_warning_flag,followup_sla_breach_flag,patient_contact_risk_flag,max_days_overdue,max_contact_attempts,latest_followup_status,latest_followup_action,complaint_count,unresolved_complaint_flag,serious_complaint_flag,angry_complaint_flag,complaint_escalated_flag,repeated_complaint_flag,patient_complaint_count,patient_repeated_complaint_flag,ticket_count,unresolved_ticket_flag,ticket_sla_breach_flag,slow_resolution_flag,ticket_escalated_flag,max_hours_open,latest_ticket_status,total_slots,booked_slots,blocked_slots,average_slot_utilization,overbooked_shift_count,slot_pressure_flag,followup_record_missing_flag
0,APP000001,BK100001,PT000808,DR0226,CL055,2025-03-22,11:30,11:50,2025-03-13 10:30:00,Clinic Desk,video,completed,checked_in,NaN,499,499,paid,Central,no,4,doctor_delay,False,True,False,False,False,Not Missed,False,False,False,False,False,False,Kriti Ghosh,30,Female,Pune,Telugu,New Patient,Thyroid,high,2022-02-15,XXXXXX3766,patient808@synthetic.example,Active,Dr. Tara Kulkarni,Gastroenterologist,Jaipur,30,3.78,0.0,False,False,False,False,False,NaN,NaN,<NA>,NaN,0.0,False,False,False,False,False,1.0,False,0.0,False,False,False,False,NaN,<NA>,68.0,56.0,2.0,86.805,0.0,True,True
1,APP000002,BK100002,PT000433,DR0563,CL043,2025-04-29,20:45,21:15,2025-04-23 09:45:00,Partner Referral,video,confirmed,,NaN,399,399,paid,Central,no,1,normal,False,False,True,True,False,Likely Missed Appointment,False,False,False,False,False,False,Kabir Menon,34,Other,Kolkata,Kannada,Regular,Cardiac Follow-up,medium,2023-10-13,XXXXXX1746,patient433@synthetic.example,Active,Dr. Aadhya Patel,Endocrinologist,Delhi NCR,15,4.34,0.0,False,False,False,False,False,NaN,NaN,<NA>,NaN,1.0,False,False,False,False,False,2.0,False,0.0,False,False,False,False,NaN,<NA>,84.0,56.0,9.0,75.378,0.0,True,True
2,APP000003,BK100003,PT000630,DR0408,CL001,2025-02-22,20:00,20:30,2025-02-07 08:00:00,Web,video,confirmed,not_checked_in,NaN,499,499,paid,North,no,0,patient_confusion,False,False,True,True,False,Likely Missed Appointment,False,False,False,False,False,False,Aditi Iyer,57,Other,Lucknow,Tamil,Chronic Care,ENT,low,2024-08-12,XXXXXX6623,patient630@synthetic.example,Active,Dr. Ishaan Mehta,Dermatologist,Pune,15,4.17,0.0,False,False,False,False,False,NaN,NaN,<NA>,NaN,0.0,False,False,False,False,False,2.0,False,0.0,False,False,False,False,NaN,<NA>,52.0,45.0,1.0,90.140,1.0,True,True
3,APP000004,BK100004,PT000374,DR0024,CL039,2025-03-12,19:00,19:15,2025-02-21 18:00:00,Call Center,hybrid,completed,checked_in,NaN,299,299,paid,Central,no,0,NaN,False,True,False,False,False,Not Missed,False,False,False,False,False,False,Ishaan Sethi,55,Female,Chennai,Bengali,Chronic Care,Gastroenterology,low,2023-06-11,XXXXXX3489,patient374@synthetic.example,Active,Dr. Ayaan Khan,Endocrinologist,Jaipur,10,4.44,1.0,False,True,False,True,True,525.0,4.0,pending,Confirm doctor slot,1.0,False,False,False,False,False,3.0,False,1.0,True,True,False,True,46.0,in_progress,76.0,48.0,8.0,69.162,0.0,True,False
4,APP000005,BK100005,PT000383,DR0341,CL018,2025-01-11,18:00,18:15,2025-01-08 12:00:00,Web,video,rescheduled,,Doctor unavailable,699,699,paid,North,n

# 12. Follow-up Priority Classification

Priority is rule-based and follows a hierarchy: High first, then Medium, Low, and finally No Issue.

One High example is a high-risk patient who missed an appointment and also has an overdue follow-up.

Another High example is an SLA-breached ticket together with an angry or escalated complaint.

Repeated no-shows with an unresolved ticket can also become High.

Medium covers cases such as overdue follow-ups without severe escalation.

The hierarchy prevents the same case from receiving conflicting classifications.

In [ ]:
def classify_priority(r):
    risk=str(r.get("risk_level","")).lower(); prev=r.get("previous_no_show_count",0); prev=0 if pd.isna(prev) else prev
    missed=bool(r.get("missed_flag",False)); overdue=bool(r.get("overdue_followup_flag",False)); complete=bool(r.get("followup_completed_flag",False))
    ut=bool(r.get("unresolved_ticket_flag",False)); sla=bool(r.get("ticket_sla_breach_flag",False)); sc=bool(r.get("serious_complaint_flag",False)); ac=bool(r.get("angry_complaint_flag",False)); ce=bool(r.get("complaint_escalated_flag",False)); uc=bool(r.get("unresolved_complaint_flag",False)); slot=bool(r.get("slot_pressure_flag",False)); st=str(r.get("appointment_status",""))
    if risk=="high" and missed and overdue: return "High"
    if sla and (ac or ce): return "High"
    if prev>=2 and ut: return "High"
    if sc and not complete: return "High"
    if overdue: return "Medium"
    if bool(r.get("followup_count",0)) and (ut or uc): return "Medium"
    if slot and st=="rescheduled": return "Medium"
    if risk=="medium" and missed: return "Medium"
    if bool(r.get("followup_count",0)) and not overdue and not ut and not sc: return "Low"
    if st=="completed" and bool(r.get("complaint_count",0)) and not uc: return "Low"
    if complete and not ut and not uc: return "Low"
    return "No Issue"
master["priority"]=master.apply(classify_priority,axis=1)
display(master["priority"].value_counts().rename_axis("priority").reset_index(name="count"))

,priority,count
0,No Issue,2362
1,Medium,1232
2,High,465
3,Low,456


# 13. Patient Impact Score

The priority category and impact score are separate.

The impact score gives a numeric severity value between zero and one hundred.

Risk level, missed appointment, overdue follow-up, unresolved ticket, SLA breach, serious complaint, and escalation contribute points.

The PRD requires a 0–100 score but does not define exact numerical weights, so I made the weights explicit and understandable.

The score is capped at 100.

In [ ]:
def impact_score(r):
    score=0; risk=str(r.get("risk_level","")).lower()
    score += 20 if risk=="high" else 10 if risk=="medium" else 0
    score += 20 if r.get("missed_flag",False) else 0
    score += 15 if r.get("overdue_followup_flag",False) else 0
    score += 10 if r.get("unresolved_ticket_flag",False) else 0
    score += 15 if r.get("ticket_sla_breach_flag",False) else 0
    score += 10 if r.get("serious_complaint_flag",False) else 0
    score += 10 if (r.get("ticket_escalated_flag",False) or r.get("complaint_escalated_flag",False)) else 0
    return min(score,100)
master["patient_impact_score"]=master.apply(impact_score,axis=1)
display(master[["appointment_id","patient_id","priority","patient_impact_score"]].head(20))

,appointment_id,patient_id,priority,patient_impact_score
0,APP000001,PT000808,No Issue,20
1,APP000002,PT000433,Medium,30
2,APP000003,PT000630,No Issue,20
3,APP000004,PT000374,Medium,50
4,APP000005,PT000383,Medium,15
5,APP000006,PT000399,Medium,45
6,APP000007,PT000954,No Issue,0
7,APP000008,PT000101,No Issue,25
8,APP000009,PT000557,No Issue,0
9,APP000010,PT000448,No Issue,0


# 14. Recommended Action Generator

Detection alone is not enough, so I convert analysis into a business action.

A High missed-appointment case can recommend urgent patient contact and appointment coordination.

A ticket SLA breach can recommend immediate escalation.

Medium cases receive standard follow-up or service-resolution actions.

No Issue cases require no immediate intervention.

This makes the output useful to an operations executive rather than only technical.

In [ ]:
def recommended_action(r):
    if r["priority"]=="High":
        if r.get("ticket_sla_breach_flag",False): return "Escalate to support/clinic operations immediately and contact patient."
        if r.get("serious_complaint_flag",False): return "Contact patient urgently and escalate complaint to clinic operations."
        if r.get("missed_flag",False): return "Call patient urgently, complete follow-up, and coordinate next appointment slot."
        return "Urgent manual review and patient contact."
    if r["priority"]=="Medium":
        if r.get("slot_pressure_flag",False): return "Coordinate doctor slot and confirm reschedule with patient."
        if r.get("overdue_followup_flag",False): return "Complete overdue patient follow-up."
        if r.get("unresolved_ticket_flag",False) or r.get("unresolved_complaint_flag",False): return "Contact patient and resolve the open service issue."
        return "Review case in standard follow-up queue."
    if r["priority"]=="Low": return "Perform routine follow-up and close after successful confirmation."
    return "No immediate operational action required."
master["recommended_action"]=master.apply(recommended_action,axis=1)
display(master[["appointment_id","priority","recommended_action"]].head(20))

,appointment_id,priority,recommended_action
0,APP000001,No Issue,No immediate operational action required.
1,APP000002,Medium,Coordinate doctor slot and confirm reschedule ...
2,APP000003,No Issue,No immediate operational action required.
3,APP000004,Medium,Coordinate doctor slot and confirm reschedule ...
4,APP000005,Medium,Complete overdue patient follow-up.
5,APP000006,Medium,Coordinate doctor slot and confirm reschedule ...
6,APP000007,No Issue,No immediate operational action required.
7,APP000008,No Issue,No immediate operational action required.
8,APP000009,No Issue,No immediate operational action required.
9,APP000010,No Issue,No immediate operational action required.


# 15. AI-Ready Support Prompt Generator

This does not call an AI API.

It only prepares structured text that could later be sent to an AI system.

Prompts are generated only for High and Medium cases.

They include operational facts such as appointment, priority, risk flags, and recommended action.

The prompt explicitly tells the future AI system not to invent medical facts.

In [ ]:
def support_prompt(r):
    if r["priority"] not in ["High","Medium"]: return ""
    return f"""ROLE: Clinic patient-support assistant
CASE PRIORITY: {r.get('priority','')}
PATIENT: {r.get('full_name','')} ({r.get('patient_id','')})
APPOINTMENT: {r.get('appointment_id','')} on {r.get('appointment_date','')}
DOCTOR: {r.get('doctor_name','')} / {r.get('specialty','')}
RISK LEVEL: {r.get('risk_level','')}
MISSED: {r.get('missed_flag',False)}
OVERDUE FOLLOW-UP: {r.get('overdue_followup_flag',False)}
UNRESOLVED TICKET: {r.get('unresolved_ticket_flag',False)}
SLA BREACH: {r.get('ticket_sla_breach_flag',False)}
UNRESOLVED COMPLAINT: {r.get('unresolved_complaint_flag',False)}
RECOMMENDED ACTION: {r.get('recommended_action','')}
TASK: Prepare a concise support summary and next operational steps. Do not invent clinical facts."""
master["ai_ready_support_prompt"]=master.apply(support_prompt,axis=1)
display(master[master["priority"].isin(["High","Medium"])][["appointment_id","priority","ai_ready_support_prompt"]].head())

,appointment_id,priority,ai_ready_support_prompt
1,APP000002,Medium,ROLE: Clinic patient-support assistant\nCASE P...
3,APP000004,Medium,ROLE: Clinic patient-support assistant\nCASE P...
4,APP000005,Medium,ROLE: Clinic patient-support assistant\nCASE P...
5,APP000006,Medium,ROLE: Clinic patient-support assistant\nCASE P...
10,APP000011,Medium,ROLE: Clinic patient-support assistant\nCASE P...


# 16. Patient Search GUI

This search box is designed for non-technical support staff.

“The user can search using patient ID, appointment ID, or patient name.

In [ ]:
search_box=widgets.Text(description="Search:",placeholder="Patient ID, Appointment ID, or Name",layout=widgets.Layout(width="70%"))
search_button=widgets.Button(description="Search",button_style="primary")
search_output=widgets.Output()
def run_search(_):
    q=search_box.value.strip().lower()
    with search_output:
        clear_output()
        if not q: print("Enter a search value."); return
        result=master[master["patient_id"].astype("string").str.lower().str.contains(q,na=False)|master["appointment_id"].astype("string").str.lower().str.contains(q,na=False)|master["full_name"].astype("string").str.lower().str.contains(q,na=False)]
        display(result[[x for x in ["appointment_id","patient_id","full_name","appointment_date","doctor_name","priority","patient_impact_score","recommended_action"] if x in result.columns]])
search_button.on_click(run_search)
display(search_box,search_button,search_output)

Text(value='', description='Search:', layout=Layout(width='70%'), placeholder='Patient ID, Appointment ID, or …

Button(button_style='primary', description='Search', style=ButtonStyle())

Output()

# 17. Filter-Based Operations View

Operations teams may not know a specific patient, so I also provide filters.

Users can filter by priority, follow-up state, specialty, city, risk, ticket condition, or complaint condition.

For example, I can immediately show all High-priority cases.

This converts the notebook into a basic operational work queue.

In [ ]:
def opts(col):
    return ["All"]+sorted(master[col].dropna().astype(str).unique().tolist()) if col in master.columns else ["All"]
priority_w=widgets.Dropdown(description="Priority",options=opts("priority")); followup_w=widgets.Dropdown(description="Follow-up",options=opts("latest_followup_status")); specialty_w=widgets.Dropdown(description="Specialty",options=opts("specialty")); city_w=widgets.Dropdown(description="City",options=opts("clinic_city")); risk_w=widgets.Dropdown(description="Risk",options=opts("risk_level")); ticket_w=widgets.Dropdown(description="Ticket",options=["All","Unresolved","SLA Breached"]); complaint_w=widgets.Dropdown(description="Complaint",options=["All","Unresolved","Escalated"]); filter_button=widgets.Button(description="Apply Filters"); filter_output=widgets.Output()
def apply_filter(_):
    r=master.copy()
    for w,col in [(priority_w,"priority"),(followup_w,"latest_followup_status"),(specialty_w,"specialty"),(city_w,"clinic_city"),(risk_w,"risk_level")]:
        if w.value!="All" and col in r.columns: r=r[r[col].astype(str)==str(w.value)]
    if ticket_w.value=="Unresolved": r=r[r.get("unresolved_ticket_flag",False)==True]
    if ticket_w.value=="SLA Breached": r=r[r.get("ticket_sla_breach_flag",False)==True]
    if complaint_w.value=="Unresolved": r=r[r.get("unresolved_complaint_flag",False)==True]
    if complaint_w.value=="Escalated": r=r[r.get("complaint_escalated_flag",False)==True]
    with filter_output:
        clear_output(); print("Filtered records:",len(r)); display(r[[x for x in ["appointment_id","patient_id","full_name","doctor_name","specialty","clinic_city","risk_level","priority","patient_impact_score","recommended_action"] if x in r.columns]].head(200))
filter_button.on_click(apply_filter)
display(priority_w,followup_w,specialty_w,city_w,risk_w,ticket_w,complaint_w,filter_button,filter_output)

Dropdown(description='Priority', options=('All', 'High', 'Low', 'Medium', 'No Issue'), value='All')

Dropdown(description='Follow-up', options=('All', '', 'closed', 'completed', 'in_progress', 'no_response', 'ov…

Dropdown(description='Specialty', options=('All', 'Cardiologist', 'Dentist', 'Dermatologist', 'ENT Specialist'…

Dropdown(description='City', options=('All', '  Bengaluru  ', '  Chennai  ', '  Kolkata  ', '  Lucknow  ', '  …

Dropdown(description='Risk', options=('All', '', 'high', 'low', 'medium'), value='All')

Dropdown(description='Ticket', options=('All', 'Unresolved', 'SLA Breached'), value='All')

Dropdown(description='Complaint', options=('All', 'Unresolved', 'Escalated'), value='All')

Button(description='Apply Filters', style=ButtonStyle())

Output()

# 18. Appointment Health Summary

This gives management-level health metrics.

It shows total appointments, completed, missed, cancelled, rescheduled, pending/confirmed, missed rate, overdue follow-ups, and High-priority cases.

This is useful for quickly understanding overall clinic performance.

In [ ]:
s=master["appointment_status"]; total=len(master); missed=int(master["missed_flag"].sum())
appointment_health_summary=pd.DataFrame([{
"total_appointments":total,"completed":int((s=="completed").sum()),"missed":missed,"cancelled":int(s.isin(["cancelled","canceled"]).sum()),"rescheduled":int((s=="rescheduled").sum()),"pending_confirmed":int(s.isin(["pending","confirmed","booked","scheduled"]).sum()),"missed_rate_pct":round(missed/total*100,2) if total else 0,"overdue_followup_count":int(master.get("overdue_followup_flag",pd.Series(False,index=master.index)).sum()),"high_priority_count":int((master["priority"]=="High").sum())}])
display(appointment_health_summary)

,total_appointments,completed,missed,cancelled,rescheduled,pending_confirmed,missed_rate_pct,overdue_followup_count,high_priority_count
0,4515,2121,1397,537,315,881,30.94,817,465


# 19. Clinic, Doctor, Patient and Ticket Summary Reports

The detailed master table is useful for case-level work, while these summaries support management analysis.

Doctor summary shows appointment load, missed appointments, High-priority cases, and average impact.

City summary helps compare operational health across clinic locations.

Patient summary identifies patients with repeated issues or high impact.

The ticket-level report remains available for support-team investigation.

In [ ]:
doctor_summary=master.groupby(["doctor_id","doctor_name","specialty","clinic_city"],dropna=False).agg(appointment_count=("appointment_id","count"),missed_count=("missed_flag","sum"),high_priority_count=("priority",lambda x:int((x=="High").sum())),average_impact_score=("patient_impact_score","mean")).reset_index()
doctor_summary["average_impact_score"]=doctor_summary["average_impact_score"].round(2)
city_summary=master.groupby("clinic_city",dropna=False).agg(appointment_count=("appointment_id","count"),missed_count=("missed_flag","sum"),high_priority_count=("priority",lambda x:int((x=="High").sum())),average_impact_score=("patient_impact_score","mean")).reset_index()
city_summary["missed_rate_pct"]=(city_summary["missed_count"]/city_summary["appointment_count"]*100).round(2)
patient_summary=master.groupby(["patient_id","full_name","city","risk_level"],dropna=False).agg(appointment_count=("appointment_id","count"),missed_count=("missed_flag","sum"),high_priority_count=("priority",lambda x:int((x=="High").sum())),maximum_impact_score=("patient_impact_score","max")).reset_index()
print("Doctor Summary"); display(doctor_summary)
print("City Summary"); display(city_summary)
print("Patient Summary"); display(patient_summary)

Doctor Summary


,doctor_id,doctor_name,specialty,clinic_city,appointment_count,missed_count,high_priority_count,average_impact_score
0,DR0001,Dr. Ayaan Agarwal,Psychiatrist,Jaipur,4,1,1,18.75
1,DR0002,Dr. Saanvi Das,General Physician,Jaipur,7,0,0,20.00
2,DR0003,Dr. Reyansh Singh,Neurologist,Hyderabad,10,6,2,23.00
3,DR0004,Dr. Kabir Verma,General Physician,Ahmedabad,11,4,1,25.91
4,DR0005,Dr. Kavya Mehta,Gastroenterologist,Delhi NCR,8,2,0,21.88
...,...,...,...,...,...,...,...,...
644,DR0646,Dr. Anika Verma,General Physician,Delhi NCR,6,2,2,25.00
645,DR0647,Dr. Reyansh Agarwal,Neurologist,Bengaluru,7,3,2,25.00
646,DR0648,Dr. Aarohi Mishra,Pediatrician,Hyderabad,9,2,0,20.00
647,DR0649,Dr. Riya Gupta,Cardiologist,Pune,3,2,0,28.33


City Summary


,clinic_city,appointment_count,missed_count,high_priority_count,average_impact_score,missed_rate_pct
0,Bengaluru,6,2,0,21.666667,33.33
1,Chennai,9,4,0,16.111111,44.44
2,Kolkata,5,1,0,15.000000,20.00
3,Lucknow,15,6,1,24.333333,40.00
4,Mumbai,13,4,0,16.538462,30.77
5,Pune,24,9,1,22.500000,37.50
6,AHMEDABAD,6,3,0,25.833333,50.00
7,Ahmedabad,350,109,27,20.514286,31.14
8,BENGALURU,16,3,1,21.875000,18.75
9,Bengaluru,416,122,37,20.564904,29.33


Patient Summary


,patient_id,full_name,city,risk_level,appointment_count,missed_count,high_priority_count,maximum_impact_score
0,PT000001,Saanvi Reddy,Pune,low,6,1,1,20
1,PT000002,Priya Reddy,Lucknow,low,1,0,0,0
2,PT000003,Kriti Das,Chennai,high,5,1,0,45
3,PT000004,Aarohi Ghosh,Bengaluru,medium,1,0,0,35
4,PT000005,Kriti Mishra,Mumbai,low,2,0,0,15
...,...,...,...,...,...,...,...,...
1171,PT001196,Anika Das,Chennai,low,7,2,2,50
1172,PT001197,Arnav Sethi,Delhi NCR,medium,7,2,2,60
1173,PT001198,Diya Malhotra,Chennai,medium,3,3,0,65
1174,PT001199,Riya Kulkarni,Hyderabad,low,3,1,1,25


# 20. Final Report Export

All useful results are exported into a dedicated output folder.

This includes the master table, summaries, anomaly report, validation report, follow-up report, complaint report, ticket report, and priority queue.

“This means business users do not need to inspect Python code to consume the results.

In [ ]:
output_folder="practo_clinic_followup_outputs"; os.makedirs(output_folder,exist_ok=True)
reports={"appointment_master.csv":master,"appointment_health_summary.csv":appointment_health_summary,"doctor_summary.csv":doctor_summary,"city_summary.csv":city_summary,"patient_summary.csv":patient_summary,"anomaly_report.csv":anomaly_report,"validation_report.csv":validation_report}
if not followups_clean.empty: reports["followup_report.csv"]=followups_clean
if not complaints_clean.empty: reports["complaint_report.csv"]=complaints_clean
if not service_tickets_clean.empty: reports["ticket_report.csv"]=service_tickets_clean
if not doctor_availability_clean.empty: reports["doctor_availability_report.csv"]=doctor_availability_clean
priority_queue=master[master["priority"].isin(["High","Medium"])].copy(); reports["priority_followup_queue.csv"]=priority_queue
reports["ai_ready_support_prompts.csv"]=priority_queue[["appointment_id","patient_id","priority","ai_ready_support_prompt"]]
for name,df in reports.items(): df.to_csv(os.path.join(output_folder,name),index=False)
print("Exported files:")
for name in sorted(os.listdir(output_folder)): print("-",name)

Exported files:
- ai_ready_support_prompts.csv
- anomaly_report.csv
- appointment_health_summary.csv
- appointment_master.csv
- city_summary.csv
- complaint_report.csv
- doctor_availability_report.csv
- doctor_summary.csv
- followup_report.csv
- patient_summary.csv
- priority_followup_queue.csv
- ticket_report.csv
- validation_report.csv


# 21. Debug Log and Code Traceability

I maintain a debug log instead of only fixing problems silently.

Examples include missing files, schema mismatches, bad relationships, appointment duplication during merges, and overlapping priority rules.

For each issue I document the section, issue type, root cause, fix, and testing status.

This makes the notebook easier to review and maintain.

In [ ]:
debug_log=pd.DataFrame([
{"issue_id":"DBG-001","section":"Loading","issue_type":"File check","description":"Required/optional files checked","root_cause":"Missing files possible","fix_summary":"Critical files stop; optional files warn","tested_status":"Tested","remarks":""},
{"issue_id":"DBG-002","section":"Validation","issue_type":"Schema","description":"Required and extra columns checked","root_cause":"Source schema variation","fix_summary":"Validation report created","tested_status":"Tested","remarks":""},
{"issue_id":"DBG-003","section":"Missed appointments","issue_type":"Business rule","description":"Explicit and likely misses handled","root_cause":"Status variants","fix_summary":"Statuses standardized","tested_status":"Tested","remarks":""},
{"issue_id":"DBG-004","section":"Linking","issue_type":"One-to-many merge","description":"Follow-up/ticket/complaint rows aggregated first","root_cause":"Direct merges duplicate appointments","fix_summary":"Aggregated before master-table merge","tested_status":"Tested","remarks":""},
{"issue_id":"DBG-005","section":"Priority","issue_type":"Business rule","description":"Priority hierarchy applied","root_cause":"Rules overlap","fix_summary":"High evaluated before Medium and Low","tested_status":"Tested","remarks":""}
])
debug_log.to_csv(os.path.join(output_folder,"debug_log.csv"),index=False)
display(debug_log)

,issue_id,section,issue_type,description,root_cause,fix_summary,tested_status,remarks
0,DBG-001,Loading,File check,Required/optional files checked,Missing files possible,Critical files stop; optional files warn,Tested,
1,DBG-002,Validation,Schema,Required and extra columns checked,Source schema variation,Validation report created,Tested,
2,DBG-003,Missed appointments,Business rule,Explicit and likely misses handled,Status variants,Statuses standardized,Tested,
3,DBG-004,Linking,One-to-many merge,Follow-up/ticket/complaint rows aggregated first,Direct merges duplicate appointments,Aggregated before master-table merge,Tested,
4,DBG-005,Priority,Business rule,Priority hierarchy applied,Rules overlap,High evaluated before Medium and Low,Tested,


# 22. Final Verification

At the end I verify that the pipeline generated the expected outputs.

I check master-table row count and unique appointment IDs.

I verify every impact score remains between zero and one hundred.

I inspect the priority distribution.

I verify that exported reports actually exist.

Edge cases: “I considered missing values, invalid dates, duplicate IDs, orphan relationships, inconsistent statuses, High-priority cases, No-Issue cases, and invalid searches.

In [ ]:
print("Master rows:",len(master))
print("Unique appointment IDs:",master["appointment_id"].nunique())
print("Impact scores valid:",master["patient_impact_score"].between(0,100).all())
print("Priority counts:",master["priority"].value_counts().to_dict())
print("Output files:",len(os.listdir(output_folder)))
if len(master)==master["appointment_id"].nunique(): print("Final master table has one row per appointment ID.")
else: print("Warning: duplicate appointment IDs remain in final master table.")
print("Implementation complete.")

Master rows: 4515
Unique appointment IDs: 4500
Impact scores valid: True
Priority counts: {'No Issue': 2362, 'Medium': 1232, 'High': 465, 'Low': 456}
Output files: 14
Implementation complete.


# Final Project Explanation

The Practo Clinic Appointment Follow-up Analyzer converts fragmented clinic operational data into one appointment-level decision-support system.

The notebook:
- loads and validates clinic datasets
- standardizes inconsistent values
- detects missed appointments
- identifies delayed follow-ups and SLA breaches
- analyzes doctor availability
- links complaints and service tickets
- detects duplicate and anomalous records
- creates one appointment-level master table
- assigns operational priority
- calculates a patient impact score
- recommends next actions
- generates AI-ready prompts
- provides search and filtering
- generates management summaries
- exports operational reports
- maintains validation and debugging information

The final result helps clinic operations teams identify cases requiring attention without manually comparing multiple files.

# Limitations and Future Improvements

## Current Limitations
- Runs inside Google Colab.
- Uses file-based input rather than a live production database.
- Does not connect to Practo production systems.
- Does not send messages or automatically contact patients.
- Does not use external AI models.
- Uses rule-based operational logic.

## Future Improvements
- Connect to a production database.
- Add real-time appointment event ingestion.
- Add role-based clinic dashboards.
- Add automated notification workflows.
- Add stronger automated tests.
- Add configurable business rules and SLA thresholds.
- Integrate a controlled AI assistant for operational summaries.